## Purpose of this notebook

This notebook walks you through:

* creating a vector database
* creating embeddings
* creating a query
* finding the closet documents
* generating a response

It is written in a notebook to provide junior data scientists and environment they are comfortable using.

### Creating a table in your database

Before starting make sure you have your postgres running using `database/docker-compose.yml`. To do this you can run from the root directory `docker compose -f database/docker-compose.yml up`

Next we will create a table. There are several embedding dimension options. As we are using gemini (`gemini-embedding-001`)for the demo you can use 768, 1536, or 3,072. The larger the bigger the cost to create and store. There are multi-modal embedding models available, but we do not use those here.

#### Set arguments for entire notebook

In [69]:
db_kwargs = {
    "host": "localhost",
    "port": 5432,
    "dbname": "rag_testing",
    "user": "bryan",
    "password": "bryan_rocks",
}

EMBEDDING_MODEL = "gemini-embedding-001"
ANSWER_MODEL = "gemini-2.5-flash"
EMBEDDING_DIMS = 1536


#### Create the table using sql

In [3]:
import psycopg

In [22]:
conn = psycopg.connect(**db_kwargs)

In [ ]:
conn.execute(f"""
    CREATE TABLE IF NOT EXISTS documents (
        id BIGSERIAL PRIMARY KEY,
        document_name TEXT NOT NULL,
        page INTEGER NOT NULL,
        text TEXT NOT NULL,
        embedding vector({EMBEDDING_DIMS})
    );
""")

conn.commit()
conn.close()

#### Ingest documents and store in table

Now that the table is created we will ingest the documents. First we will insert the text into documents. For the news article I will create chuncks for "pages", otherwise for the FM we will just use pages.

I first insert the text so that if the API call fails, we still have the text in the database. Also batching the embeddings is cheaper. For this model (`gemini-embedding-001`) it costs `$0.075` for `1M` tokens (Its `$0.15` for `1M` tokens if you do not batch).

In [13]:
from pathlib import Path
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [10]:
files = [str(f) for f in Path("data").glob("**/*.txt")]

Statement for inserting documents

In [12]:
insert_sql = """
    INSERT INTO documents (document_name, page, text, embedding)
    VALUES (%s, %s, %s, NULL)
"""

Used for chunking the news articles. This will have each "page" be 500 tokens with zero overlap. These are two parameters you can use to adjust your chunking strategy to see how it impacts your results.

In [20]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=0,
)

Insert the text

In [24]:
with psycopg.connect(**db_kwargs) as conn:
    with conn.cursor() as cur:
        for file_path in files:
            path = Path(file_path)
            text = path.read_text()

            if "fm" in str(file_path):
                page = int(file_path.split("_")[-1].replace(".txt", ""))
                cur.execute(insert_sql, (path.name, page, text))
            else:
                chunks = splitter.split_text(text)
                start_page = 1

                for i, chunk in enumerate(chunks):
                    cur.execute(
                        insert_sql,
                        (path.name, start_page + i, chunk),
                    )

**Create and insert the embeddings**  If you have not done so already, you need to add your api key to a `.api_key` in the root directory.

In [66]:
import numpy as np
import pandas as pd
from google import genai
from google.genai import types
from pgvector.psycopg import register_vector

In [41]:
with open(".api_key", "r") as f:
    api_key = f.read()

**Query the text from the DB**

In [37]:
conn = psycopg.connect(**db_kwargs)
cur = conn.cursor()
cur.execute(
            """
            SELECT id, text
            FROM documents
            WHERE embedding IS NULL
            ORDER BY id
            """
        )
rows = cur.fetchall()
cur.close()
conn.close()

##### Generate the embeddings and insert them

There is a lot that goes on here so I will explain it as I go. First you need to turn your documents (the text on each page) into a list of documents. You then send it to get your embeddings. Our embedding model and dimension require us to normalize the embeddings. Lastly you need to update the table.

In [63]:
# turn documents into a list of documents
document_ids = [row[0] for row in rows]
documents = [row[1] for row in rows]

In [ ]:
EMBEDDING_DIMS

1536

In [ ]:
# create your config for the embeddings
config = types.EmbedContentConfig(
    task_type="RETRIEVAL_DOCUMENT",
    output_dimensionality=EMBEDDING_DIMS,
)

In [57]:
def batch_embeddings(api_key:str, documents: list[str], config: types.EmbedContentConfig, model:str = "gemini-embedding-001"):
    """
    Embed a list of documents in batches, the api limits you to 100 documents per request.
    """

    all_embeddings = []

    for i in range(0, len(documents), 100):
        chunk = documents[i : i + 100]
        
        response = client.models.embed_content(
            model=model,
            contents=chunk,
            config=config,
        )
        
        # aggregate your embeddings
        all_embeddings.extend(response.embeddings)

    return all_embeddings

In [ ]:
text_embeddings = batch_embeddings(api_key, documents, config, EMBEDDING_MODEL)

In [62]:
# normalize the embeddings for inserting into the database
raw_embeddings = np.array([e.values for e in text_embeddings])
norms = np.linalg.norm(raw_embeddings, axis=1, keepdims=True)
normalized_embeddings = raw_embeddings / norms

In [64]:
# prep data for inserting into the table
insert_rows = list(zip(normalized_embeddings, document_ids))

In [65]:
# insert the rows
conn = psycopg.connect(**db_kwargs)
register_vector(conn)
cur = conn.cursor()
cur.executemany(
    """
    UPDATE documents
    SET embedding = %s
    WHERE id = %s
    """,
    insert_rows,
)
conn.commit()
cur.close()
conn.close()

In [68]:
# check the table

conn = psycopg.connect(**db_kwargs)
cur = conn.cursor()
cur.execute("""
    SELECT id, document_name, page, text, embedding
    FROM documents
    ORDER BY id
    LIMIT 10
""")

rows = cur.fetchall()
cur.close()
conn.close()

df = pd.DataFrame(rows, columns=["id", "document_name", "page", "text", "embedding"])
df.head()



,id,document_name,page,text,embedding
0,1,page_127.txt,127,Offense \n11 January 2024 ATP 3-21.8 (INCL C1)...,"[-0.01985496,-0.0024588972,-0.008230229,-0.075..."
1,2,page_133.txt,133,Offense \n11 January 2024 ATP 3-21.8 (INCL C1)...,"[-0.008289794,-0.013305798,-0.0038737203,-0.07..."
2,3,page_132.txt,132,Chapter 4 \n4-10 ATP 3-21.8 (INCL C1) 11 Janua...,"[-0.014288951,-0.009373881,0.01363416,-0.06900..."
3,4,page_126.txt,126,Chapter 4 \n4-4 ATP 3-21.8 (INCL C1) 11 Januar...,"[-0.027320085,0.026910396,-0.014111335,-0.0656..."
4,5,page_130.txt,130,Chapter 4 \n4-8 ATP 3-21.8 (INCL C1) 11 Januar...,"[-0.014810689,-0.022852749,0.0010444222,-0.060..."


## Now the question and answer part

Now the fun begins. You now have your database of your documents, or really the facts you would like to use to answer your questions. Now we can ask questions. There are a lot of settings you can pick, but I will just go through a few. I am going to make them into functions, this reason why will become more obvious as we transition to tooling.

In [79]:
class RAGService:
    def __init__(
        self,
        db_kwargs: dict,
        api_key: str,
        table_name: str = "documents",
        k: int = 3,
        answer_model: str = "gemini-2.5-flash",
        embed_model: str = "gemini-embedding-001",
        embedding_dims: int = 1536,
        instructions: str = "",
    ):
        self.db_kwargs = db_kwargs
        self.table_name = table_name
        self.k = k
        self.answer_model = answer_model
        self.embed_model = embed_model
        self.instructions = instructions
        self.embedding_dims = embedding_dims
        self.client = genai.Client(api_key=api_key)

    def _normalize(self, vec):
        arr = np.asarray(vec, dtype=np.float32)
        norm = np.linalg.norm(arr)
        return (arr / norm).tolist() if norm else arr.tolist()

    def embed_query(self, question: str):
        resp = self.client.models.embed_content(
            model=self.embed_model,
            contents=question,
            config=types.EmbedContentConfig(
                task_type="QUESTION_ANSWERING",
                output_dimensionality=self.embedding_dims,
            ),
        )
        if self.embedding_dims != 3072:
            self._normalize(resp.embeddings[0].values)
        return resp.embeddings[0].values

    def get_top_k_docs(self, question: str):
        query_embedding = self.embed_query(question)

        with psycopg.connect(**self.db_kwargs) as conn:
            register_vector(conn)
            with conn.cursor() as cur:
                cur.execute(
                    f"""
                    SELECT document_name, page, text
                    FROM {self.table_name}
                    ORDER BY embedding <#> %s::vector
                    LIMIT %s
                    """,
                    (query_embedding, self.k),
                )
                return cur.fetchall()

    def build_context(self, docs):
        return "\n\n".join(
            f"[{document_name} page {page}]\n{text}"
            for document_name, page, text in docs
        )

    def answer_question(self, question: str) -> str:
        docs = self.get_top_k_docs(question)
        context = self.build_context(docs)

        prompt = f"""{self.instructions}

                Context:
                {context}

                Question:
                {question}

                """

        resp = self.client.models.generate_content(
            model=self.answer_model,
            contents=prompt,
        )
        return resp.text or ""

In [91]:
instructions = """
You are a retrieval-augmented question answering assistant.

Rules:
- Answer only from the provided context.
- If the context does not contain the answer, say so clearly.
- Keep answers concise and direct.
- Do not invent facts or cite unsupported details.
- Prefer quoting or paraphrasing the source text when useful.
- If the user asks for analysis, explain your reasoning briefly.
"""


In [81]:
rag_service = RAGService(
    db_kwargs=db_kwargs,
    api_key=api_key,
    table_name="documents",
    k=3,
    answer_model=ANSWER_MODEL,
    embed_model=EMBEDDING_MODEL,
    embedding_dims=EMBEDDING_DIMS,
    instructions=instructions,
)

In [82]:
rag_service.answer_question("who is winning the Ukraine war?")

'The provided context presents a mixed and somewhat contradictory picture regarding who is winning the Ukraine war.\n\n*   By late November (presumably 2022, as it precedes 2023-2024 events), Ukraine had reclaimed approximately 74,000sq km, reducing Russian control to about 19 percent of the country, following a withdrawal from Kherson city and gains in Kharkiv oblast. (aljazeera.txt page 4)\n*   From 2023, the conflict became one of attrition, with Russian forces taking Soledar and Bakhmut after brutal combat, and Avdiivka in 2024. (aljazeera.txt page 4)\n*   More recently, "For the first time in years, according to Kyiv, Ukraine had liberated more territory than it lost to Russian occupation in February thanks to a series of counterattacking operations on the southern front line," and President Volodymyr Zelensky stated on March 16th that Ukrainian forces disrupted Russia\'s plans for an offensive operation over March. (kyivindependent.txt page 2, 3)\n*   However, "Russian forces are

In [83]:
rag_service.answer_question("Explain to me audacity.")

"Audacity is a willingness to take bold risks. The bold execution of plans increases the chance for surprise and allows leaders to control the tempo. It depends on a leader's ability to identify opportunities and accept prudent risks that align with the value of their objectives. Leaders demonstrate audacity by acting decisively to dispel uncertainty, inspiring Soldiers, and compensating for a lack of information by aggressively developing the situation and seizing the initiative.\n\nThe Infantry platoon and squad demonstrate audacity through:\n*   Building flexible plans.\n*   Maintaining situational awareness.\n*   Understanding the true value of an objective and associated risks.\n*   Maintaining continuous communications.\n*   Requesting additional support as needed."

Now look what happens when I change the instructions.

In [84]:
instructions = """
You are a retrieval-augmented question answering assistant.

Rules:
- Answer only from the provided context.
- If the context does not contain the answer, say so clearly.
- Keep answers concise and direct.
- Do not invent facts or cite unsupported details.
- Prefer quoting or paraphrasing the source text when useful.
- If the user asks for analysis, explain your reasoning briefly.
- only pay attention to articles from 2022
"""

In [85]:
rag_service = RAGService(
    db_kwargs=db_kwargs,
    api_key=api_key,
    table_name="documents",
    k=3,
    answer_model=ANSWER_MODEL,
    embed_model=EMBEDDING_MODEL,
    embedding_dims=EMBEDDING_DIMS,
    instructions=instructions,
)

In [86]:
rag_service.answer_question("who is winning the Ukraine war?")

'By late November 2022, Ukraine had reclaimed approximately 74,000sq km (28,600sq miles) of territory, reducing Russian control to about 19 percent of the country.'

In [87]:
rag_service.answer_question("Is ukraine winning the war, yes or no?")

'The provided context from 2022 indicates that Ukraine reclaimed approximately 74,000 sq km (28,600 sq miles) of territory and forced a withdrawal from Kherson city by late November, reducing Russian control to about 19 percent of the country. However, the context does not explicitly state that Ukraine is "winning the war."'